# Incident Response Runbook: Command and Control - Remote Access Tools

**Tactic:** Command and Control
**Technique:** Remote Access Tools (T1219)
**Severity:** CRITICAL

## Overview

This runbook provides step-by-step incident response procedures for Remote Access Tools activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime, timedelta
import sys
import os

# Add parent directory to path for imports
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_data_collector import CrowdStrikeDataCollector
from misp.misp_integration import MISPIntegration
from iris.iris_integration import IRISIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Step 1: Initial Alert Analysis & Detection
print("=" * 60)
print("STEP 1: Initial Alert Analysis & Detection")
print("=" * 60)

# Initialize security integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeDataCollector()
misp = MISPIntegration()
iris = IRISIntegration()
shuffle = ShuffleIntegration()

# Create incident case in IRIS
incident_data = {
    'title': 'Remote Access Tools C2 Activity - T1219',
    'description': 'Potential remote access tool command and control activity detected',
    'severity': 'HIGH',
    'category': 'C2 Activity',
    'tags': ['remote-access-tools', 'c2', 'rat', 'command-control', 't1219']
}

case_id = iris.create_case(incident_data)
print(f" Created IRIS case: {case_id}")

# Detection queries for remote access tools
print("\n Executing remote access tool detection queries...")

# Splunk query for known RAT signatures
rat_query = """
index=* sourcetype=network_traffic OR sourcetype=endpoint_logs
| where (dest_port IN (3389, 5900, 5901, 22, 23, 513, 514) AND protocol="TCP")
OR (ImageFileName =~ "*teamviewer*" OR ImageFileName =~ "*anydesk*" OR ImageFileName =~ "*chrome remote*" OR ImageFileName =~ "*logmein*")
OR (process_name =~ "*vnc*" OR process_name =~ "*rdp*" OR process_name =~ "*ssh*" OR process_name =~ "*telnet*")
| where NOT (user IN ("administrator", "system", "networkservice"))
| stats count by host, dest_ip, dest_port, process_name, user
| where count > 2
"""

rat_results = splunk.execute_query(rat_query)
print(f" Found {len(rat_results)} potential RAT activities")

# Splunk query for suspicious C2 communications
c2_comm_query = """
index=* sourcetype=network_traffic
| where (dest_port IN (4444, 5555, 6666, 7777, 8888, 9999, 12345, 31337))
OR (dest_ip IN (misp_indicators{:c2_servers}))
| stats count by src_ip, dest_ip, dest_port, protocol
| where count > 10
| sort -count
"""

c2_results = splunk.execute_query(c2_comm_query)
print(f" Found {len(c2_results)} suspicious C2 communications")

# CrowdStrike query for RAT process behavior
cs_rat_query = """
event_simpleName=ProcessRollup2
| where ImageFileName =~ "*vnc*" OR ImageFileName =~ "*teamviewer*" OR ImageFileName =~ "*anydesk*"
OR ImageFileName =~ "*logmein*" OR ImageFileName =~ "*gotomypc*" OR ImageFileName =~ "*splashtop*"
OR CommandLine =~ "*reverse*" OR CommandLine =~ "*tunnel*" OR CommandLine =~ "*proxy*"
| stats count by ComputerName, UserName, ImageFileName, CommandLine
"""

cs_rat_results = crowdstrike.execute_query(cs_rat_query)
print(f" Found {len(cs_rat_results)} RAT process activities")

# Splunk query for unauthorized remote access installations
unauthorized_rat_query = """
index=* sourcetype=software_inventory OR sourcetype=installation_logs
| where software_name =~ "*teamviewer*" OR software_name =~ "*anydesk*" OR software_name =~ "*logmein*"
OR software_name =~ "*vnc*" OR software_name =~ "*rdp*" OR software_name =~ "*remote desktop*"
| where NOT approved=true
| stats count by host, software_name, installer_user
"""

unauthorized_results = splunk.execute_query(unauthorized_rat_query)
print(f" Found {len(unauthorized_results)} unauthorized RAT installations")

# Check for known RAT IOCs in MISP
rat_indicators = misp.search_indicators("remote-access-tools", limit=50)
print(f" Retrieved {len(rat_indicators)} RAT indicators from MISP")

# Compile detection results
detection_summary = {
    'case_id': case_id,
    'timestamp': datetime.now().isoformat(),
    'tactic': 'Command and Control',
    'technique': 'Remote Access Tools (T1219)',
    'severity': 'HIGH',
    'indicators': {
        'rat_activities': len(rat_results),
        'c2_communications': len(c2_results),
        'rat_processes': len(cs_rat_results),
        'unauthorized_installations': len(unauthorized_results),
        'misp_indicators': len(rat_indicators)
    },
    'affected_hosts': list(set([r.get('host', r.get('ComputerName', 'unknown')) 
                               for r in rat_results + cs_rat_results + unauthorized_results])),
    'suspicious_connections': len(c2_results)
}

print(f"\n📊 Detection Summary:")
print(f"  Case ID: {detection_summary['case_id']}")
print(f"  Severity: {detection_summary['severity']}")
print(f"  Affected Hosts: {len(detection_summary['affected_hosts'])}")
print(f"  RAT Activities: {detection_summary['indicators']['rat_activities']}")
print(f"  C2 Communications: {detection_summary['indicators']['c2_communications']}")
print(f"  RAT Processes: {detection_summary['indicators']['rat_processes']}")
print(f"  Unauthorized Installations: {detection_summary['indicators']['unauthorized_installations']}")

# Trigger Shuffle workflow for automated RAT response
if detection_summary['indicators']['rat_activities'] > 0 or detection_summary['indicators']['c2_communications'] > 0:
    shuffle.trigger_workflow('rat_detection_response', detection_summary)
    print(" Triggered automated RAT response workflow")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment Actions
print("\n" + "=" * 60)
print("STEP 2: Containment Actions")
print("=" * 60)

containment_actions = []

# Block suspicious C2 connections
print("\n Blocking suspicious C2 connections...")
for connection in c2_results[:100]:  # Limit to top 100 to avoid overload
    try:
        block_result = shuffle.block_ip(connection['dest_ip'])
        if block_result.get('success'):
            containment_actions.append(f" Blocked C2 IP: {connection['dest_ip']}:{connection['dest_port']}")
            print(f"   Blocked C2 IP: {connection['dest_ip']}:{connection['dest_port']}")
    except Exception as e:
        containment_actions.append(f" Failed to block IP {connection['dest_ip']}: {str(e)}")
        print(f"   Failed to block IP {connection['dest_ip']}: {str(e)}")

# Isolate affected hosts via CrowdStrike
print("\n Isolating affected hosts...")
for host in detection_summary['affected_hosts']:
    try:
        isolation_result = crowdstrike.isolate_host(host)
        if isolation_result.get('success'):
            containment_actions.append(f" Isolated host: {host}")
            print(f"   Isolated host: {host}")
        else:
            containment_actions.append(f" Failed to isolate host: {host}")
            print(f"   Failed to isolate host: {host}")
    except Exception as e:
        containment_actions.append(f" Error isolating {host}: {str(e)}")
        print(f"   Error isolating {host}: {str(e)}")

# Disable remote access services
print("\n Disabling remote access services...")
for host in detection_summary['affected_hosts']:
    try:
        disable_result = shuffle.disable_remote_access(host)
        if disable_result.get('success'):
            containment_actions.append(f" Disabled remote access on: {host}")
            print(f"   Disabled remote access on: {host}")
    except Exception as e:
        containment_actions.append(f" Failed to disable remote access on {host}: {str(e)}")
        print(f"   Failed to disable remote access on {host}: {str(e)}")

# Block unauthorized RAT software
print("\n🛑 Blocking unauthorized RAT software...")
for software in unauthorized_results:
    try:
        block_software_result = shuffle.block_software(software['software_name'])
        if block_software_result.get('success'):
            containment_actions.append(f" Blocked software: {software['software_name']}")
            print(f"   Blocked software: {software['software_name']}")
    except Exception as e:
        containment_actions.append(f" Failed to block software {software['software_name']}: {str(e)}")
        print(f"   Failed to block software {software['software_name']}: {str(e)}")

# Enable network segmentation
print("\n🌐 Enabling network segmentation...")
try:
    segmentation_config = {
        'isolate_affected_hosts': True,
        'block_remote_access_ports': True,
        'enable_micro_segmentation': True,
        'duration_hours': 24
    }
    segmentation_result = shuffle.enable_network_segmentation(segmentation_config)
    if segmentation_result.get('success'):
        containment_actions.append(" Network segmentation enabled")
        print("   Network segmentation enabled")
except Exception as e:
    containment_actions.append(f" Failed to enable network segmentation: {str(e)}")
    print(f"   Failed to enable network segmentation: {str(e)}")

# Update IRIS case with containment actions
iris.update_case(case_id, {
    'containment_actions': containment_actions,
    'containment_timestamp': datetime.now().isoformat(),
    'status': 'containment_in_progress'
})

print(f"\n Containment Summary:")
print(f"  Actions Taken: {len([a for a in containment_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in containment_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in containment_actions if a.startswith('')])}")

# Trigger Shuffle workflow for containment verification
shuffle.trigger_workflow('rat_containment_verification', {
    'case_id': case_id,
    'actions_taken': containment_actions
})
print(" Triggered containment verification workflow")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication Actions
print("\n" + "=" * 60)
print("STEP 3: Eradication Actions")
print("=" * 60)

eradication_actions = []

# Terminate RAT processes via CrowdStrike
print("\n💀 Terminating RAT processes...")
for process in cs_rat_results:
    try:
        kill_result = crowdstrike.terminate_process(process['ComputerName'], process['ProcessId'])
        if kill_result.get('success'):
            eradication_actions.append(f" Terminated RAT process: {process['ImageFileName']} on {process['ComputerName']}")
            print(f"   Terminated RAT process: {process['ImageFileName']} on {process['ComputerName']}")
    except Exception as e:
        eradication_actions.append(f" Failed to terminate process {process['ProcessId']}: {str(e)}")
        print(f"   Failed to terminate process {process['ProcessId']}: {str(e)}")

# Remove RAT software and files
print("\n🗑 Removing RAT software and files...")
for software in unauthorized_results:
    try:
        removal_result = shuffle.uninstall_software(software['host'], software['software_name'])
        if removal_result.get('success'):
            eradication_actions.append(f" Removed software: {software['software_name']} from {software['host']}")
            print(f"   Removed software: {software['software_name']} from {software['host']}")
    except Exception as e:
        eradication_actions.append(f" Failed to remove software {software['software_name']}: {str(e)}")
        print(f"   Failed to remove software {software['software_name']}: {str(e)}")

# Clean up persistence mechanisms
print("\n Cleaning up persistence mechanisms...")
for host in detection_summary['affected_hosts']:
    try:
        persistence_result = crowdstrike.remove_persistence(host)
        if persistence_result.get('success'):
            eradication_actions.append(f" Cleaned persistence on: {host}")
            print(f"   Cleaned persistence on: {host}")
    except Exception as e:
        eradication_actions.append(f" Failed to clean persistence on {host}: {str(e)}")
        print(f"   Failed to clean persistence on {host}: {str(e)}")

# Reset remote access configurations
print("\n🔧 Resetting remote access configurations...")
for host in detection_summary['affected_hosts']:
    try:
        reset_result = shuffle.reset_remote_access_config(host)
        if reset_result.get('success'):
            eradication_actions.append(f" Reset remote access config on: {host}")
            print(f"   Reset remote access config on: {host}")
    except Exception as e:
        eradication_actions.append(f" Failed to reset config on {host}: {str(e)}")
        print(f"   Failed to reset config on {host}: {str(e)}")

# Update firewall rules
print("\n Updating firewall rules...")
try:
    firewall_rules = {
        'block_rat_ports': True,
        'block_suspicious_outbound': True,
        'enable_strict_remote_access': True,
        'rat_ports': [3389, 5900, 5901, 22, 23, 513, 514, 4444, 5555, 6666, 7777, 8888, 9999, 12345, 31337]
    }
    firewall_result = shuffle.update_firewall_rules(firewall_rules)
    if firewall_result.get('success'):
        eradication_actions.append(" Firewall rules updated")
        print("   Firewall rules updated")
except Exception as e:
    eradication_actions.append(f" Failed to update firewall rules: {str(e)}")
    print(f"   Failed to update firewall rules: {str(e)}")

# Verify threat removal
print("\n✅ Verifying threat removal...")
verification_results = []
for host in detection_summary['affected_hosts']:
    try:
        scan_result = crowdstrike.perform_threat_scan(host)
        if scan_result.get('threats_found', 0) == 0:
            verification_results.append(f" Clean scan on {host}")
            print(f"   Clean scan on {host}")
        else:
            verification_results.append(f" Threats still present on {host}: {scan_result.get('threats_found')}")
            print(f"   Threats still present on {host}: {scan_result.get('threats_found')}")
    except Exception as e:
        verification_results.append(f" Scan failed on {host}: {str(e)}")
        print(f"   Scan failed on {host}: {str(e)}")

# Update IRIS case with eradication actions
iris.update_case(case_id, {
    'eradication_actions': eradication_actions,
    'verification_results': verification_results,
    'eradication_timestamp': datetime.now().isoformat(),
    'status': 'eradication_complete'
})

print(f"\n Eradication Summary:")
print(f"  Actions Taken: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Verification Results: {len([r for r in verification_results if r.startswith('')])} clean hosts")

# Share indicators with MISP
if len(rat_indicators) > 0:
    misp.share_indicators(rat_indicators, case_id)
    print(" Shared RAT indicators with MISP community")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery Actions
print("\n" + "=" * 60)
print("STEP 4: Recovery Actions")
print("=" * 60)

recovery_actions = []

# Re-enable isolated hosts
print("\n🔓 Re-enabling isolated hosts...")
for host in detection_summary['affected_hosts']:
    try:
        reenable_result = crowdstrike.reenable_host(host)
        if reenable_result.get('success'):
            recovery_actions.append(f" Re-enabled host: {host}")
            print(f"   Re-enabled host: {host}")
        else:
            recovery_actions.append(f" Failed to re-enable host: {host}")
            print(f"   Failed to re-enable host: {host}")
    except Exception as e:
        recovery_actions.append(f" Error re-enabling {host}: {str(e)}")
        print(f"   Error re-enabling {host}: {str(e)}")

# Restore approved remote access capabilities
print("\n🔑 Restoring approved remote access...")
for host in detection_summary['affected_hosts']:
    try:
        restore_result = shuffle.restore_approved_remote_access(host)
        if restore_result.get('success'):
            recovery_actions.append(f" Restored approved remote access on: {host}")
            print(f"   Restored approved remote access on: {host}")
    except Exception as e:
        recovery_actions.append(f" Failed to restore remote access on {host}: {str(e)}")
        print(f"   Failed to restore remote access on {host}: {str(e)}")

# Validate system integrity
print("\n Validating system integrity...")
integrity_checks = []
for host in detection_summary['affected_hosts']:
    try:
        integrity_result = crowdstrike.validate_system_integrity(host)
        if integrity_result.get('integrity_valid'):
            integrity_checks.append(f" System integrity valid on {host}")
            print(f"   System integrity valid on {host}")
        else:
            integrity_checks.append(f" Integrity issues on {host}: {integrity_result.get('issues', [])}")
            print(f"   Integrity issues on {host}: {integrity_result.get('issues', [])}")
    except Exception as e:
        integrity_checks.append(f" Integrity check failed on {host}: {str(e)}")
        print(f"   Integrity check failed on {host}: {str(e)}")

# Implement secure remote access policies
print("\n Implementing secure remote access policies...")
try:
    policy_config = {
        'require_mfa': True,
        'session_timeout': 480,  # 8 hours
        'approved_tools_only': True,
        'logging_enabled': True,
        'monitoring_enabled': True
    }
    policy_result = shuffle.implement_remote_access_policies(policy_config)
    if policy_result.get('success'):
        recovery_actions.append(" Secure remote access policies implemented")
        print("   Secure remote access policies implemented")
except Exception as e:
    recovery_actions.append(f" Failed to implement remote access policies: {str(e)}")
    print(f"   Failed to implement remote access policies: {str(e)}")

# Monitor for RAT recurrence
print("\n Establishing monitoring for RAT recurrence...")
try:
    recurrence_monitoring = {
        'hosts': detection_summary['affected_hosts'],
        'duration_days': 30,
        'alert_threshold': 'medium',
        'indicators': ['rat_processes', 'suspicious_connections', 'unauthorized_software']
    }
    splunk.setup_recurrence_monitoring(recurrence_monitoring)
    recovery_actions.append(" Recurrence monitoring established")
    print("   Recurrence monitoring established")
except Exception as e:
    recovery_actions.append(f" Failed to establish recurrence monitoring: {str(e)}")
    print(f"   Failed to establish recurrence monitoring: {str(e)}")

# Conduct security assessment
print("\n🔬 Conducting security assessment...")
try:
    assessment_config = {
        'scan_for_rat_software': True,
        'check_remote_access_configs': True,
        'validate_firewall_rules': True,
        'audit_user_permissions': True
    }
    assessment_result = shuffle.conduct_security_assessment(assessment_config)
    if assessment_result.get('success'):
        recovery_actions.append(" Security assessment completed")
        print("   Security assessment completed")
except Exception as e:
    recovery_actions.append(f" Security assessment failed: {str(e)}")
    print(f"   Security assessment failed: {str(e)}")

# Update IRIS case with recovery actions
iris.update_case(case_id, {
    'recovery_actions': recovery_actions,
    'integrity_checks': integrity_checks,
    'recovery_timestamp': datetime.now().isoformat(),
    'status': 'recovery_complete'
})

print(f"\n Recovery Summary:")
print(f"  Actions Taken: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Integrity Checks Passed: {len([c for c in integrity_checks if c.startswith('')])}")

# Trigger Shuffle workflow for recovery verification
shuffle.trigger_workflow('rat_recovery_verification', {
    'case_id': case_id,
    'actions_taken': recovery_actions,
    'integrity_checks': integrity_checks
})
print(" Triggered recovery verification workflow")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident Actions
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []

# Generate incident report
print("\n Generating incident report...")
try:
    incident_report = {
        'case_id': case_id,
        'title': 'Remote Access Tools C2 Incident Response Report',
        'timeline': {
            'detection': detection_summary['timestamp'],
            'containment': datetime.now().isoformat(),  # Would be actual timestamp
            'eradication': datetime.now().isoformat(),   # Would be actual timestamp
            'recovery': datetime.now().isoformat(),      # Would be actual timestamp
            'closure': datetime.now().isoformat()
        },
        'impact_assessment': {
            'affected_hosts': len(detection_summary['affected_hosts']),
            'c2_connections': detection_summary['indicators']['c2_communications'],
            'unauthorized_software': detection_summary['indicators']['unauthorized_installations'],
            'business_impact': 'HIGH',  # Would be determined by business impact analysis
            'data_exposure': 'POTENTIAL',  # Would be determined by investigation
            'persistence_risk': 'HIGH'  # RATs often establish strong persistence
        },
        'response_metrics': {
            'detection_to_containment': 'TBD',  # Would calculate actual time
            'containment_to_eradication': 'TBD',
            'total_resolution_time': 'TBD',
            'automation_level': 'HIGH'
        },
        'attack_characteristics': {
            'attack_vector': 'Remote Access Tools',
            'c2_communications': detection_summary['indicators']['c2_communications'],
            'rat_processes': detection_summary['indicators']['rat_processes'],
            'unauthorized_installations': detection_summary['indicators']['unauthorized_installations'],
            'persistence_mechanisms': 'MULTIPLE'
        },
        'lessons_learned': [
            'Unauthorized remote access tools pose significant risk',
            'Automated process termination prevented further compromise',
            'Network segmentation contained the threat effectively',
            'MISP integration provided valuable RAT intelligence',
            'Host isolation was critical for containing C2 activity'
        ]
    }

    report_id = iris.generate_report(case_id, incident_report)
    post_incident_actions.append(f" Generated incident report: {report_id}")
    print(f"   Generated incident report: {report_id}")

except Exception as e:
    post_incident_actions.append(f" Failed to generate report: {str(e)}")
    print(f"   Failed to generate report: {str(e)}")

# Document lessons learned
print("\n Documenting lessons learned...")
lessons_learned = [
    "Implement strict remote access tool approval processes",
    "Regular auditing of installed software and remote access configurations",
    "Advanced endpoint detection for RAT behaviors",
    "Network monitoring for suspicious C2 patterns",
    "User training on safe remote access practices",
    "Regular security assessments for remote access infrastructure"
]

try:
    iris.add_lessons_learned(case_id, lessons_learned)
    post_incident_actions.append(" Documented lessons learned")
    print("   Documented lessons learned")
except Exception as e:
    post_incident_actions.append(f" Failed to document lessons learned: {str(e)}")
    print(f"   Failed to document lessons learned: {str(e)}")

# Implement preventive measures
print("\n Implementing preventive measures...")
preventive_measures = []

try:
    # Update remote access security policies
    policy_updates = {
        'rat_detection': 'enhanced',
        'remote_access_monitoring': 'continuous',
        'software_whitelisting': True,
        'network_segmentation': 'strict'
    }
    shuffle.update_security_policies(policy_updates)
    preventive_measures.append(" Updated remote access security policies")
    print("   Updated remote access security policies")

    # Enhance monitoring rules
    monitoring_updates = {
        'rat_behavior_detection': True,
        'c2_communication_monitoring': True,
        'unauthorized_software_detection': True
    }
    splunk.update_monitoring_rules(monitoring_updates)
    preventive_measures.append(" Enhanced monitoring rules")
    print("   Enhanced monitoring rules")

    # Update threat intelligence feeds
    misp.update_feeds(['remote-access-tools', 'rat', 'c2-servers'])
    preventive_measures.append(" Updated threat intelligence feeds")
    print("   Updated threat intelligence feeds")

except Exception as e:
    preventive_measures.append(f" Failed to implement preventive measures: {str(e)}")
    print(f"   Failed to implement preventive measures: {str(e)}")

# Conduct post-incident review
print("\n Conducting post-incident review...")
try:
    review_findings = {
        'effectiveness_rating': 'HIGH',
        'areas_for_improvement': [
            'Earlier detection of unauthorized RAT installations',
            'Better inventory management for remote access tools',
            'Enhanced user training on RAT risks'
        ],
        'recommendations': [
            'Implement automated software inventory scanning',
            'Regular remote access security assessments',
            'Advanced behavioral analysis for endpoint detection',
            'Zero-trust network architecture',
            'Automated policy enforcement for remote access'
        ]
    }

    iris.conduct_post_incident_review(case_id, review_findings)
    post_incident_actions.append(" Conducted post-incident review")
    print("   Conducted post-incident review")

except Exception as e:
    post_incident_actions.append(f" Failed to conduct review: {str(e)}")
    print(f"   Failed to conduct review: {str(e)}")

# Close the incident case
print("\n Closing incident case...")
try:
    closure_data = {
        'closure_reason': 'Remote access tools C2 activity contained - RATs removed, access secured',
        'closure_timestamp': datetime.now().isoformat(),
        'final_status': 'CLOSED',
        'follow_up_required': False,
        'reopen_criteria': 'Detection of similar RAT C2 activity'
    }

    iris.close_case(case_id, closure_data)
    post_incident_actions.append(" Closed incident case")
    print("   Closed incident case")

except Exception as e:
    post_incident_actions.append(f" Failed to close case: {str(e)}")
    print(f"   Failed to close case: {str(e)}")

# Final summary
print(f"\n Post-Incident Summary:")
print(f"  Case ID: {case_id}")
print(f"  Actions Completed: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Preventive Measures: {len(preventive_measures)}")
print(f"  Lessons Learned: {len(lessons_learned)}")

print("\n Incident Response Complete")
print("=" * 60)
print("Remote access tools C2 incident successfully contained and resolved.")
print("RATs removed, access secured, and preventive measures implemented.")
print("=" * 60)


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
